In [1]:
# Cell 1
from finvizfinance.earnings import Earnings
import pandas as pd
from datetime import datetime, timedelta
import re


In [2]:
# Cell 2
# Initialize Earnings object and load data
e = Earnings()
df = e.df.copy()

# Inspect columns if needed
df.head()


/Users/devna/personalProjects/financeApp/venv/lib/python3.13/site-packages/finvizfinance/screener/base.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(frame)], ignore_index=True)


,Ticker,Market Cap,Dividend,ROA,ROE,ROIC,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M,Oper M,Profit M,Earnings,Price,Change,Volume
0,ACDC,690920000.0,NaN,-0.1120,-0.3146,-17.14%,0.91,0.63,1.10,1.30,0.0394,-0.0671,-0.1706,Nov 10/b,3.82,0.0324,2054861.0
1,ADCT,490550000.0,NaN,-0.5226,NaN,-76.47%,4.63,4.34,NaN,NaN,0.8933,-1.6294,-2.2197,Nov 10/b,3.96,-0.0222,560424.0
2,AGEN,147600000.0,NaN,-0.1489,NaN,-,0.47,0.47,NaN,NaN,0.8708,-0.5892,-0.3312,Nov 10/b,4.34,0.0023,364360.0
3,AIOT,658410000.0,NaN,-0.0490,-0.0898,-5.83%,1.07,0.93,0.51,0.61,0.5462,0.0266,-0.0970,Nov 10/b,4.92,-0.0140,1278326.0
4,AKBA,443160000.0,NaN,-0.0557,NaN,-7.21%,1.94,1.80,4.30,5.13,0.7655,0.0788,-0.0707,Nov 10/b,1.67,-0.0511,13597026.0


In [20]:
def parse_market_cap(mc):
    """Convert Market Cap string like '1.2T', '500B', '30M' to float for sorting"""
    if pd.isna(mc):
        return 0
    mc = str(mc).upper().strip()
    if mc.endswith('T'):
        return float(mc[:-1]) * 1_000_000_000_000
    elif mc.endswith('B'):
        return float(mc[:-1]) * 1_000_000_000
    elif mc.endswith('M'):
        return float(mc[:-1]) * 1_000_000
    elif mc.endswith('K'):
        return float(mc[:-1]) * 1_000
    else:
        try:
            return float(mc)
        except:
            return 0

df['Market Cap Numeric'] = df['Market Cap'].apply(parse_market_cap)


In [21]:
def parse_earnings_date(s):
    if pd.isna(s):
        return None
    match = re.match(r'([A-Za-z]+) (\d{1,2})', s)
    if not match:
        return None
    month_str, day = match.groups()
    try:
        date = datetime.strptime(f"{month_str} {day} {datetime.today().year}", "%b %d %Y")
        if date < datetime.today():
            date = date.replace(year=date.year + 1)
        return date
    except:
        return None

df['Earnings Date'] = df['Earnings'].apply(parse_earnings_date)


In [22]:
def format_market_cap(n):
    """Convert numeric Market Cap to string with T/B/M/K"""
    if n >= 1_000_000_000_000:
        return f"{n/1_000_000_000_000:.2f}T"
    elif n >= 1_000_000_000:
        return f"{n/1_000_000_000:.2f}B"
    elif n >= 1_000_000:
        return f"{n/1_000_000:.2f}M"
    elif n >= 1_000:
        return f"{n/1_000:.2f}K"
    else:
        return str(n)


In [23]:
# This week's companies

# Apply formatting after sorting
df_sorted = df.sort_values('Market Cap Numeric', ascending=False)
top10 = df_sorted.head(20)
top10['Market Cap Formatted'] = top10['Market Cap Numeric'].apply(format_market_cap)

# Display final table
top10[['Ticker', 'Market Cap Formatted', 'Earnings']]

/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_15938/3987648552.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top10['Market Cap Formatted'] = top10['Market Cap Numeric'].apply(format_market_cap)


,Ticker,Market Cap Formatted,Earnings
399,CSCO,308.35B,Nov 12/a
528,DIS,190.22B,Nov 13/b
804,MUFG,180.32B,Nov 14/a
615,AMAT,180.05B,Nov 13/a
226,SONY,180.03B,Nov 11/b
808,SMFG,111.56B,Nov 14/a
518,BN,109.03B,Nov 13/b
803,MFG,87.26B,Nov 14/a
223,SE,76.97B,Nov 11/b
686,NU,76.23B,Nov 13/a


In [30]:
e = Earnings(period='nextweek')
df_next = e.df.copy()


ValueError: Invalid period 'nextweek'. Available period: ['This Week', 'Next Week', 'Previous Week', 'This Month']